# Qwen2.5-3B Baseline Eval — Disaster Response

Runs the **untuned** Qwen2.5-3B-Instruct model against all 3 difficulties.
Saves scores, per-step logs, and a bar chart to `/content/`.

Run order: Cell 0 → 1 → 2 → 3 → 4 → 5 → 6

In [ ]:
%%capture
# Clone repo (skips .env and all .gitignore'd files automatically)
!git clone https://github.com/MSArunkutz/disaster-env /content/disaster_env

# Install env dependencies
!pip install -q -r /content/disaster_env/requirements.txt 2>/dev/null || \
    pip install -q fastapi uvicorn pydantic python-dotenv openai websockets httpx

# Install training dependencies
!pip install unsloth transformers accelerate bitsandbytes matplotlib

In [ ]:
import os, sys, json, re
from pathlib import Path

DISASTER_ENV_PATH = Path("/content/disaster_env")
sys.path.insert(0, str(DISASTER_ENV_PATH))

MODEL_NAME   = "unsloth/Qwen2.5-3B-Instruct"
MAX_SEQ_LEN  = 2048
LOAD_IN_4BIT = True
MAX_STEPS    = 200

OUTPUT_DIR   = Path("/content")
LOG_FILE     = OUTPUT_DIR / "baseline_log.json"
RESULTS_FILE = OUTPUT_DIR / "baseline_results.json"
CHART_FILE   = OUTPUT_DIR / "baseline_chart.png"

print("disaster_env path:", DISASTER_ENV_PATH.resolve())
print("Exists:", DISASTER_ENV_PATH.exists())
print("Config loaded.")

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = LOAD_IN_4BIT,
)
FastLanguageModel.for_inference(model)
print("Loaded (untuned):", MODEL_NAME)

In [ ]:
def extract_action_json(text: str):
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if m:
        try: return json.loads(m.group(1))
        except: pass
    m = re.search(r"(\{.*\})", text, re.DOTALL)
    if m:
        try: return json.loads(m.group(1))
        except: pass
    return None

In [ ]:
from server.disaster_env_environment import DisasterEnvEnvironment
from models import DisasterAction
from graders import compute_score


def run_baseline_episode(difficulty: str) -> tuple:
    """Run one episode, return (score, step_logs)."""
    env = DisasterEnvEnvironment(difficulty=difficulty)
    obs = env.reset()
    step_logs = []

    for _ in range(MAX_STEPS):
        if obs.done:
            break

        prompt_text = (
            f"Step {obs.current_step}/{obs.max_steps}\n"
            f"{obs.zones_summary}\n{obs.resources_summary}\n{obs.available_actions}\n"
            "Respond with a JSON deployments object."
        )
        inputs  = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True)
        text    = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        action_data = extract_action_json(text)
        entry = {
            "step":          obs.current_step,
            "raw_output":    text[:400],
            "parse_success": action_data is not None,
        }

        if action_data and "deployments" in action_data:
            try:
                obs = env.step(DisasterAction(**action_data))
            except Exception as e:
                entry["error"] = str(e)
                step_logs.append(entry)
                break
        else:
            entry["parse_failed"] = True
            step_logs.append(entry)
            break

        step_logs.append(entry)

    score = compute_score(env)
    return score, step_logs


all_logs = {}
results  = {}

for diff in ["easy", "medium", "hard"]:
    print(f"Running {diff}...")
    score, logs = run_baseline_episode(diff)
    results[diff]  = score
    all_logs[diff] = logs
    parse_ok = sum(1 for s in logs if s.get("parse_success"))
    print(f"  {diff:8s}  score={score:.4f}  steps={len(logs)}  parsed={parse_ok}/{len(logs)}")

avg = sum(results.values()) / 3
results["average"] = round(avg, 4)
print(f"\nAverage: {avg:.4f}")

In [ ]:
with open(RESULTS_FILE, "w") as f:
    json.dump({"model": MODEL_NAME, "results": results}, f, indent=2)
print("Saved:", RESULTS_FILE)

with open(LOG_FILE, "w") as f:
    json.dump(all_logs, f, indent=2)
print("Saved:", LOG_FILE)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

difficulties = ["easy", "medium", "hard"]
scores = [results[d] for d in difficulties]

x = np.arange(len(difficulties))
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(x, scores, color="steelblue", width=0.5)
ax.set_xticks(x)
ax.set_xticklabels(difficulties)
ax.set_ylim(0, 1.05)
ax.set_ylabel("compute_score")
ax.set_title(f"Qwen2.5-3B Baseline (untuned)")
for i, s in enumerate(scores):
    ax.text(i, s + 0.01, f"{s:.4f}", ha="center", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(CHART_FILE, dpi=150)
plt.show()
print("Saved:", CHART_FILE)
print("\n--- Final Baseline Scores ---")
for d in difficulties:
    print(f"  {d:8s}: {results[d]:.4f}")
print(f"  {'average':8s}: {results['average']:.4f}")